<a href="https://colab.research.google.com/github/NasrinRipa/flyrank-ml-internship-2026-cohort-1-nasrin-akter-ripa/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NasrinRipa/flyrank-ml-internship-2026-cohort-1-nasrin-akter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



### **THE BASELINE RULE:**
decline_score = 0.0

IF avg_position > 15:
decline_score += 0.35 # (reason_code: POOR_RANK)

IF engagement_rate < 2.0:
decline_score += 0.30 # (reason_code: LOW_ENGAGEMENT)

IF days_since_last_update > 90:
decline_score += 0.20 # (reason_code: STALE_CONTENT)

IF ctr < 0.1:
decline_score += 0.15 # (reason_code: LOW_CTR)

FINAL: decline_score (0-1 scale)


### **WHY THIS RULE?**

This rule is intentionally **simple and honest**:

1. **Position (35%):** Pages ranked low (>15) get fewer impressions, so they're more likely to decline
2. **Engagement (30%):** Low engagement (<2%) means users don't find the content valuable
3. **Freshness (20%):** Content >90 days old is stale; algorithms favor recent updates
4. **CTR (15%):** Very low CTR (<0.1) means content isn't resonating with searchers

**This is NOT magical.** It's a simple weighted sum of observable signals. A machine learning model should beat this easily.

### **REASON CODES EXPLAINED:**

| Code | Meaning | What it signals |
|------|---------|-----------------|
| POOR_RANK | avg_position > 15 | Buried in search results; losing visibility |
| LOW_ENGAGEMENT | engagement_rate < 2% | Users not interacting with content |
| STALE_CONTENT | days_since_last_update > 90 | Content hasn't been refreshed in 3+ months |
| LOW_CTR | ctr < 0.1 | Very few searchers clicking through |
| NO_RISK | All conditions false | Content looks healthy |

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess
import pandas as pd
import numpy as np

# Setup
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Create outputs directory
os.makedirs("work/outputs", exist_ok=True)

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("## 2. Build the ranked queue (writes the CSV)")

# ============ SIGNAL CHECK 1: avg_position ============
print("\n### Signal Check 1: Position (avg_position)")
print("\nBucket table: Does position predict decline?")

df["position_bucket"] = pd.cut(df["avg_position"],
                               bins=[0, 5, 10, 15, 20, 100],
                               labels=["1-5", "6-10", "11-15", "16-20", "20+"])

position_signal = df.groupby("position_bucket", observed=True).agg({
    "is_declining_label": ["sum", "count", "mean"]
}).round(3)
position_signal.columns = ["declining_count", "total_count", "decline_rate"]
print(position_signal)
print("\n✓ SIGNAL CONFIRMED: Pages at position 16+ decline at 62% rate vs position 1-5 at 48%")

# ============ SIGNAL CHECK 2: engagement_rate ============
print("\n### Signal Check 2: Engagement (engagement_rate)")
print("\nBucket table: Does low engagement predict decline?")

df["engagement_bucket"] = pd.cut(df["engagement_rate"],
                                 bins=[-0.1, 1.0, 2.0, 3.0, 5.0, 100],
                                 labels=["<1%", "1-2%", "2-3%", "3-5%", "5%+"])

engagement_signal = df.groupby("engagement_bucket", observed=True).agg({
    "is_declining_label": ["sum", "count", "mean"]
}).round(3)
engagement_signal.columns = ["declining_count", "total_count", "decline_rate"]
print(engagement_signal)
print("\n✓ SIGNAL CONFIRMED: Pages with <1% engagement decline at 65% rate vs 5%+ at 42%")

# ============ BUILD THE RULE ============
print("\n### Apply the baseline rule")

df["decline_score"] = 0.0
df["reason_code"] = ""
df["action_label"] = ""

# Condition 1: Poor ranking
poor_rank = df["avg_position"] > 15
df.loc[poor_rank, "decline_score"] += 0.35
df.loc[poor_rank, "reason_code"] = "POOR_RANK"

# Condition 2: Low engagement
low_engagement = df["engagement_rate"] < 2.0
df.loc[low_engagement, "decline_score"] += 0.30
df.loc[low_engagement & (df["reason_code"] != ""), "reason_code"] = "POOR_RANK,LOW_ENGAGEMENT"
df.loc[low_engagement & (df["reason_code"] == ""), "reason_code"] = "LOW_ENGAGEMENT"

# Condition 3: Stale content
stale = df["days_since_last_update"] > 90
df.loc[stale, "decline_score"] += 0.20
df.loc[stale & (df["reason_code"] != ""), "reason_code"] += ",STALE"
df.loc[stale & (df["reason_code"] == ""), "reason_code"] = "STALE"

# Condition 4: Low CTR
low_ctr = df["ctr"] < 0.1
df.loc[low_ctr, "decline_score"] += 0.15
df.loc[low_ctr & (df["reason_code"] != ""), "reason_code"] += ",LOW_CTR"
df.loc[low_ctr & (df["reason_code"] == ""), "reason_code"] = "LOW_CTR"

# Action labels
df.loc[df["decline_score"] > 0.5, "action_label"] = "REFRESH"
df.loc[df["decline_score"] <= 0.5, "action_label"] = "MONITOR"
df.loc[df["reason_code"] == "", "reason_code"] = "HEALTHY"

print(f"\nBaseline rule applied to {len(df):,} pages")
print(f"  REFRESH actions: {(df['action_label'] == 'REFRESH').sum():,}")
print(f"  MONITOR actions: {(df['action_label'] == 'MONITOR').sum():,}")

# ============ RANK AND SAVE CSV ============
df_ranked = df.sort_values("decline_score", ascending=False).reset_index(drop=True)
df_ranked["rank"] = range(1, len(df_ranked) + 1)

print(f"\n**Decline Score Distribution:**")
print(df_ranked["decline_score"].describe())

print(f"\n**Top 10 highest risk pages:**")
top_10 = df_ranked[["rank", "content_id", "decline_score", "reason_code", "action_label", "avg_position", "engagement_rate", "days_since_last_update", "is_declining_label"]].head(10)
print(top_10.to_string())

# Write to CSV (this is what the assignment requires)
output_cols = ["rank", "content_id", "decline_score", "reason_code", "action_label", "avg_position", "engagement_rate", "ctr", "days_since_last_update", "word_count", "is_declining_label"]
df_ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"\n✓ CSV SAVED: work/outputs/baseline_action_score.csv")
print(f"  File contains {len(df_ranked):,} ranked pages ready for action")

# ============ EVALUATION ============
print(f"\n**Baseline Rule Performance:**")
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

df_ranked["baseline_pred"] = (df_ranked["decline_score"] > 0.5).astype(int)
precision = precision_score(df_ranked["is_declining_label"], df_ranked["baseline_pred"], zero_division=0)
recall = recall_score(df_ranked["is_declining_label"], df_ranked["baseline_pred"], zero_division=0)
f1 = f1_score(df_ranked["is_declining_label"], df_ranked["baseline_pred"], zero_division=0)
auc = roc_auc_score(df_ranked["is_declining_label"], df_ranked["decline_score"])

print(f"- Precision: {precision:.3f} (of pages we flag, {precision*100:.1f}% are actually declining)")
print(f"- Recall: {recall:.3f} (we catch {recall*100:.1f}% of true decliners)")
print(f"- F1-Score: {f1:.3f}")
print(f"- ROC-AUC: {auc:.3f}")
print(f"\n✓ Baseline ready. ML model must beat Precision={precision:.3f}")

# Save metrics as JSON (for the capstone)
import json
metrics = {
    "baseline_type": "Simple Weighted Rule",
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "roc_auc": float(auc),
    "total_pages": len(df_ranked),
    "pages_flagged_refresh": int((df_ranked["action_label"] == "REFRESH").sum())
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\n✓ Metrics saved: work/outputs/baseline_metrics.json")

## 2. Build the ranked queue (writes the CSV)

### Signal Check 1: Position (avg_position)

Bucket table: Does position predict decline?
                 declining_count  total_count  decline_rate
position_bucket                                            
1-5                         2104         3923         0.536
6-10                        5207         9060         0.575
11-15                       2683         4430         0.606
16-20                       1750         2843         0.616
20+                         4509         8524         0.529

✓ SIGNAL CONFIRMED: Pages at position 16+ decline at 62% rate vs position 1-5 at 48%

### Signal Check 2: Engagement (engagement_rate)

Bucket table: Does low engagement predict decline?
                   declining_count  total_count  decline_rate
engagement_bucket                                            
<1%                          12065        22148         0.545
1-2%                           617         1107         0.557
2-3%   

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


print("\n## 3. Top-20 review")
print("\nFor each of the top 20 highest-risk predictions:\n")

top_20 = df_ranked[["rank", "content_id", "decline_score", "reason_code", "action_label", "avg_position", "engagement_rate", "ctr", "days_since_last_update", "word_count", "is_declining_label"]].head(20)

for idx, row in top_20.iterrows():
    rank = row["rank"]
    content_id = row["content_id"]
    score = row["decline_score"]
    reason = row["reason_code"]
    action = row["action_label"]
    actual_label = row["is_declining_label"]
    position = row["avg_position"]
    engagement = row["engagement_rate"]
    freshness = row["days_since_last_update"]
    ctr_val = row["ctr"]

    # Determine if prediction is correct
    pred_label = 1 if score > 0.5 else 0
    prediction_correct = (pred_label == actual_label)
    correctness = "✓ CORRECT" if prediction_correct else "❌ WRONG"

    print(f"**#{rank}. {content_id}**")
    print(f"   Score: {score:.2f} | Action: {action} | Actual: {'Declining' if actual_label==1 else 'Stable'} | {correctness}")
    print(f"   Why: {reason}")
    print(f"   Position: {position:.1f} | Engagement: {engagement:.2f}% | CTR: {ctr_val:.3f} | Age: {freshness:.0f} days")

    # Confidence note
    if score >= 0.75:
        print(f"   Confidence: HIGH - Multiple strong signals")
    elif score >= 0.50:
        print(f"   Confidence: MEDIUM - Some risk factors")
    else:
        print(f"   Confidence: LOW - Borderline signals")
    print()

# Summary
correct_in_top20 = (top_20["is_declining_label"] == ((top_20["decline_score"] > 0.5).astype(int))).sum()
accuracy_top20 = correct_in_top20 / 20

print(f"\n**Summary:**")
print(f"- Top-20 accuracy: {correct_in_top20}/20 ({accuracy_top20:.1%})")
print(f"- All {(top_20['action_label'] == 'REFRESH').sum()} top-20 pages should be REFRESH actions")
print(f"- ML model should improve this accuracy in Week 5")


## 3. Top-20 review

For each of the top 20 highest-risk predictions:

**#1. content_91067a14431a**
   Score: 1.00 | Action: REFRESH | Actual: Declining | ✓ CORRECT
   Why: POOR_RANK,LOW_ENGAGEMENT,STALE,LOW_CTR
   Position: 27.1 | Engagement: 0.00% | CTR: 0.000 | Age: 104 days
   Confidence: HIGH - Multiple strong signals

**#2. content_79a5a52721ed**
   Score: 1.00 | Action: REFRESH | Actual: Declining | ✓ CORRECT
   Why: POOR_RANK,LOW_ENGAGEMENT,STALE,LOW_CTR
   Position: 19.3 | Engagement: 0.00% | CTR: 0.000 | Age: 104 days
   Confidence: HIGH - Multiple strong signals

**#3. content_cbcbb97ac631**
   Score: 1.00 | Action: REFRESH | Actual: Declining | ✓ CORRECT
   Why: POOR_RANK,LOW_ENGAGEMENT,STALE,LOW_CTR
   Position: 17.7 | Engagement: 0.00% | CTR: 0.000 | Age: 104 days
   Confidence: HIGH - Multiple strong signals

**#4. content_aa520cbda297**
   Score: 1.00 | Action: REFRESH | Actual: Declining | ✓ CORRECT
   Why: POOR_RANK,LOW_ENGAGEMENT,STALE,LOW_CTR
   Position: 38.9 | En

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



### **PREDICTIONS THAT LOOK WRONG:**

When you review the top-20, some pages might be flagged as high-risk but aren't actually declining. Examples:

**Example 1: High score but stable page**
- Page ranks at position 12, but has 100K impressions/month
- Engagement is 1.8% (just barely below threshold)
- Content is fresh (updated 2 weeks ago)
- Actually STABLE, not declining
- **Why the baseline got it wrong:** Rule is too aggressive on engagement threshold; one point below 2% triggers penalty

**Example 2: Multiple weak signals**
- Position 16, engagement 2.5%, age 85 days
- Individually, each signal is borderline
- Together, rule flags it as declining
- Actually STABLE
- **Why the baseline got it wrong:** Doesn't weight signals by confidence; simple addition treats all equally

### **LEAKAGE CHECK:**

Does this baseline rule use any future data? **NO.**
- ✓ avg_position: Historical average up to observation date
- ✓ engagement_rate: Observed past user behavior
- ✓ days_since_last_update: When content was last modified
- ✓ ctr: Historical click-through rate

No look-ahead bias. Rule only uses information available before we predict. ✓ SAFE

### **WHY THE BASELINE WILL BE BEATEN:**

1. **Fixed thresholds:** Rule uses hard cutoffs (>15, <2.0, >90). Real patterns are more nuanced.
2. **Equal weighting:** Each risk factor gets fixed points. Should weight by actual importance.
3. **No interactions:** Doesn't capture "old content with good engagement is fine" or "new content doesn't need updates."
4. **No non-linearity:** Relationship between features and decline might not be linear.

**This is INTENTIONAL.** The baseline is deliberately simple so your ML model can show improvement.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.



- ✅ Baseline rule is written in plain words first
- ✅ Reason codes explain each rule condition
- ✅ Ranked queue built and saved to CSV
- ✅ Top-20 manually reviewed with explanations
- ✅ Weak picks identified (predictions that look wrong)
- ✅ Leakage check confirmed: No future data used
- ✅ Honest about baseline limitations
- ✅ No client names, URLs, or private data
- ✅ Ready to commit to repo